# Anomaly Detection & Pattern Analysis

This notebook goes beyond viewing anomaly flags — it analyzes *why* anomalies
occur, which devices are most at risk, how z-scores correlate with threshold
violations, and what early-warning signals exist.

**Sections:**
1. Anomaly Landscape — overall rates and per-type breakdown
2. Threshold Analysis — sensor distributions with boundary visualization
3. Z-Score Patterns — statistical outlier detection vs rule-based flags
4. Temporal Patterns — anomaly trends over time
5. Device Risk Profiling — Gold-layer health scores and risk tiers

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pyspark.sql.functions import (
    col, count, avg, when, lit,
    sum as spark_sum, abs as spark_abs,
    min as spark_min, max as spark_max,
)

from pipeline.common.utils import load_config, get_spark_session

config = load_config()
spark = get_spark_session(config)

silver_df = spark.read.format("delta").load(config["paths"]["delta_silver"])
silver_pdf = silver_df.toPandas()

total = len(silver_pdf)
print(f"Loaded {total:,} Silver records")

## 1. Anomaly Landscape

How many records are flagged, and which sensor types trigger most often?

In [ ]:
anom_mask = silver_pdf["_is_anomaly"] == True
anom_count = anom_mask.sum()
clean_count = total - anom_count

type_counts = {
    "Temperature": (silver_pdf["_is_temp_anomaly"] == True).sum(),
    "Humidity":    (silver_pdf["_is_humidity_anomaly"] == True).sum(),
    "Pressure":    (silver_pdf["_is_pressure_anomaly"] == True).sum(),
}

print(f"Total anomalies: {anom_count:,} / {total:,} ({anom_count/total*100:.1f}%)")
print(f"Clean records:   {clean_count:,}")
for t, c in type_counts.items():
    print(f"  {t}: {c:,}")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Clean vs Anomalous", "Anomaly Type Breakdown"],
    specs=[[{"type": "pie"}, {"type": "bar"}]],
)

fig.add_trace(go.Pie(
    labels=["Clean", "Anomalous"],
    values=[clean_count, anom_count],
    marker_colors=["#2ecc71", "#e74c3c"],
    hole=0.4, textinfo="label+percent",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=list(type_counts.keys()),
    y=list(type_counts.values()),
    marker_color=["#e74c3c", "#3498db", "#9b59b6"],
    text=list(type_counts.values()), textposition="outside",
), row=1, col=2)

fig.update_layout(
    title="Anomaly Landscape",
    template="plotly_white", height=400, showlegend=False,
)
fig.show()

### Per-Device Anomaly Distribution

Some devices may have systematic sensor faults while others are clean.

In [ ]:
device_anom = silver_pdf.groupby("device_id").agg(
    total_events=("_is_anomaly", "count"),
    anomaly_count=("_is_anomaly", "sum"),
).reset_index()
device_anom["anomaly_rate"] = device_anom["anomaly_count"] / device_anom["total_events"]
device_anom = device_anom.sort_values("anomaly_rate", ascending=False)

top_20 = device_anom.head(20)

fig = go.Figure(go.Bar(
    x=top_20["device_id"],
    y=top_20["anomaly_rate"],
    marker_color=top_20["anomaly_rate"].apply(
        lambda r: "#e74c3c" if r > 0.5 else "#f39c12" if r > 0 else "#2ecc71"
    ),
    text=top_20["anomaly_count"].astype(int).astype(str) + "/" + top_20["total_events"].astype(str),
    textposition="outside",
))
fig.update_layout(
    title="Top 20 Devices by Anomaly Rate",
    xaxis_title="Device", yaxis_title="Anomaly Rate",
    yaxis_range=[0, 1.1],
    template="plotly_white", height=400,
)
fig.show()

## 2. Threshold Analysis

The pipeline uses rule-based thresholds for anomaly detection.
Let's visualize where records fall relative to these boundaries.

| Sensor | Min | Max |
|---|---|---|
| Temperature | -50 | 150 |
| Humidity | 0 | 100 |
| Pressure | 900 | 1100 |

In [ ]:
thresholds = {
    "temperature": {"min": -50, "max": 150, "flag": "_is_temp_anomaly"},
    "humidity":    {"min": 0,   "max": 100, "flag": "_is_humidity_anomaly"},
    "pressure":    {"min": 900, "max": 1100, "flag": "_is_pressure_anomaly"},
}

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Temperature", "Humidity", "Pressure"],
)

for i, (field, cfg) in enumerate(thresholds.items(), 1):
    clean_vals = silver_pdf[silver_pdf[cfg["flag"]] == False][field].dropna()
    anom_vals  = silver_pdf[silver_pdf[cfg["flag"]] == True][field].dropna()

    fig.add_trace(go.Histogram(
        x=clean_vals, name="Within Range", marker_color="#2ecc71",
        opacity=0.7, showlegend=(i == 1),
    ), row=1, col=i)

    if len(anom_vals) > 0:
        fig.add_trace(go.Histogram(
            x=anom_vals, name="Out of Range", marker_color="#e74c3c",
            opacity=0.7, showlegend=(i == 1),
        ), row=1, col=i)

    fig.add_vline(x=cfg["min"], line_dash="dash", line_color="orange",
                  annotation_text="Min", row=1, col=i)
    fig.add_vline(x=cfg["max"], line_dash="dash", line_color="orange",
                  annotation_text="Max", row=1, col=i)

fig.update_layout(
    title="Sensor Distributions with Anomaly Thresholds",
    barmode="overlay", template="plotly_white", height=400,
)
fig.show()

## 3. Z-Score Patterns

Z-scores measure how far each reading is from the expected population mean.
A |z| > 3 is a strong statistical signal even if the value is within hard thresholds.

This reveals a complementary view to rule-based detection: **subtle drift**
that isn't extreme enough to trigger thresholds but is statistically unusual.

In [ ]:
zscore_cols = ["_temperature_zscore", "_humidity_zscore", "_pressure_zscore"]
labels = ["Temperature", "Humidity", "Pressure"]

fig = make_subplots(rows=1, cols=3, subplot_titles=labels)

for i, (zc, label) in enumerate(zip(zscore_cols, labels), 1):
    clean = silver_pdf[~anom_mask][zc].dropna()
    anom  = silver_pdf[anom_mask][zc].dropna()

    fig.add_trace(go.Histogram(
        x=clean, name="Clean", marker_color="#2ecc71",
        opacity=0.7, nbinsx=40, showlegend=(i == 1),
    ), row=1, col=i)
    fig.add_trace(go.Histogram(
        x=anom, name="Anomalous", marker_color="#e74c3c",
        opacity=0.7, nbinsx=40, showlegend=(i == 1),
    ), row=1, col=i)

    for z_line in [-3, 3]:
        fig.add_vline(x=z_line, line_dash="dot", line_color="gray",
                      row=1, col=i)

fig.update_layout(
    title="Z-Score Distributions (dashed lines = |z| = 3 boundary)",
    barmode="overlay", template="plotly_white", height=400,
)
fig.show()

### Z-Score vs Threshold Agreement

How often do statistical outliers (|z| > 3) agree with rule-based flags?
Disagreements highlight cases where one method catches what the other misses.

In [ ]:
for field, zc, flag in [
    ("Temperature", "_temperature_zscore", "_is_temp_anomaly"),
    ("Humidity", "_humidity_zscore", "_is_humidity_anomaly"),
    ("Pressure", "_pressure_zscore", "_is_pressure_anomaly"),
]:
    z_outlier = silver_pdf[zc].abs() > 3
    rule_flag = silver_pdf[flag] == True

    both    = (z_outlier & rule_flag).sum()
    z_only  = (z_outlier & ~rule_flag).sum()
    r_only  = (~z_outlier & rule_flag).sum()
    neither = (~z_outlier & ~rule_flag).sum()

    print(f"\n{field}:")
    print(f"  Both detect:      {both:>5}")
    print(f"  Z-score only:     {z_only:>5}  (statistical outlier, within thresholds)")
    print(f"  Threshold only:   {r_only:>5}  (out of range, but not statistically extreme)")
    print(f"  Neither:          {neither:>5}")

## 4. Temporal Patterns

Are anomalies uniformly distributed or do they cluster in time?
Temporal clusters suggest environmental events rather than random sensor faults.

In [ ]:
silver_pdf["timestamp_dt"] = pd.to_datetime(silver_pdf["timestamp"], utc=True)
silver_pdf["minute"] = silver_pdf["timestamp_dt"].dt.floor("1min")

time_series = silver_pdf.groupby("minute").agg(
    total=("_is_anomaly", "count"),
    anomalies=("_is_anomaly", "sum"),
).reset_index()
time_series["anomaly_rate"] = time_series["anomalies"] / time_series["total"]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Events per Minute", "Anomaly Rate per Minute"],
    vertical_spacing=0.08,
)

fig.add_trace(go.Scatter(
    x=time_series["minute"], y=time_series["total"],
    mode="lines", name="Total Events",
    line=dict(color="#3498db"),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=time_series["minute"], y=time_series["anomaly_rate"],
    mode="lines+markers", name="Anomaly Rate",
    line=dict(color="#e74c3c"),
    marker=dict(size=4),
), row=2, col=1)

fig.add_hline(y=0.1, line_dash="dot", line_color="gray",
              annotation_text="10% warning line", row=2, col=1)

fig.update_layout(
    title="Anomaly Trends Over Time",
    template="plotly_white", height=500,
)
fig.show()

## 5. Early Warning Signals

Can we spot anomalies before they happen? Low battery and drifting z-scores
are potential leading indicators of upcoming sensor faults.

In [ ]:
fig = px.scatter(
    silver_pdf, x="battery_level", y="_temperature_zscore",
    color=silver_pdf["_is_anomaly"].map({True: "Anomalous", False: "Clean"}),
    color_discrete_map={"Clean": "#2ecc71", "Anomalous": "#e74c3c"},
    opacity=0.5,
    title="Battery Level vs Temperature Z-Score",
    labels={"battery_level": "Battery Level (%)", "_temperature_zscore": "Temp Z-Score"},
)
fig.update_layout(template="plotly_white", height=400)
fig.show()

low_batt = silver_pdf[silver_pdf["battery_level"] < 30]
high_batt = silver_pdf[silver_pdf["battery_level"] >= 30]
print(f"Anomaly rate for low-battery devices (<30%):  "
      f"{low_batt['_is_anomaly'].mean()*100:.1f}%")
print(f"Anomaly rate for healthy-battery devices:      "
      f"{high_batt['_is_anomaly'].mean()*100:.1f}%")

## 6. Device Risk Profiling (Gold Layer)

The Gold `device_health` table computes a composite health score:

```
health = 0.4 * avg_quality + 0.3 * (1 - anomaly_rate) + 0.3 * (battery / 100)
```

Devices are classified into risk tiers: **healthy** (≥0.8), **warning** (≥0.5), **critical** (<0.5).

In [ ]:
health_path = str(Path(config["paths"]["delta_gold"]) / "device_health")
health_pdf = spark.read.format("delta").load(health_path).toPandas()

tier_colors = {"healthy": "#2ecc71", "warning": "#f39c12", "critical": "#e74c3c"}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Health Score Distribution", "Risk Tier Breakdown"],
    specs=[[{"type": "histogram"}, {"type": "pie"}]],
)

for tier, color in tier_colors.items():
    subset = health_pdf[health_pdf["risk_tier"] == tier]
    if not subset.empty:
        fig.add_trace(go.Histogram(
            x=subset["health_score"], name=tier.capitalize(),
            marker_color=color, nbinsx=15,
        ), row=1, col=1)

tier_counts = health_pdf["risk_tier"].value_counts()
fig.add_trace(go.Pie(
    labels=[t.capitalize() for t in tier_counts.index],
    values=tier_counts.values,
    marker_colors=[tier_colors.get(t, "#888") for t in tier_counts.index],
    hole=0.4,
), row=1, col=2)

fig.update_layout(
    title="Device Health Overview",
    template="plotly_white", height=400, barmode="stack",
)
fig.show()

### Highest-Risk Devices

Which devices need immediate attention?

In [ ]:
at_risk = health_pdf.sort_values("health_score").head(10)

fig = go.Figure(go.Bar(
    y=at_risk["device_id"],
    x=at_risk["health_score"],
    orientation="h",
    marker_color=at_risk["risk_tier"].map(tier_colors),
    text=at_risk["risk_tier"].str.capitalize(),
    textposition="inside",
))

fig.add_vline(x=0.8, line_dash="dash", line_color="#2ecc71",
              annotation_text="Healthy threshold")
fig.add_vline(x=0.5, line_dash="dash", line_color="#f39c12",
              annotation_text="Warning threshold")

fig.update_layout(
    title="10 Lowest-Scoring Devices",
    xaxis_title="Health Score", xaxis_range=[0, 1],
    template="plotly_white", height=400,
)
fig.show()

### Health Score Decomposition

Break down what drives each at-risk device's score.

In [ ]:
at_risk_detail = at_risk.copy()
at_risk_detail["quality_component"]  = 0.4 * at_risk_detail["avg_quality_score"]
at_risk_detail["anomaly_component"]  = 0.3 * (1 - at_risk_detail["anomaly_rate"])
at_risk_detail["battery_component"]  = 0.3 * (at_risk_detail["avg_battery_level"] / 100)

fig = go.Figure()
for comp, label, color in [
    ("quality_component",  "Quality (40%)",      "#3498db"),
    ("anomaly_component",  "Anomaly-Free (30%)", "#e74c3c"),
    ("battery_component",  "Battery (30%)",      "#f39c12"),
]:
    fig.add_trace(go.Bar(
        y=at_risk_detail["device_id"],
        x=at_risk_detail[comp],
        name=label, orientation="h",
        marker_color=color,
    ))

fig.update_layout(
    title="Health Score Decomposition — At-Risk Devices",
    xaxis_title="Score Contribution",
    barmode="stack", template="plotly_white", height=400,
)
fig.show()

In [ ]:
spark.stop()